# Dynamax LGSSM Walkthrough

This notebook builds a linear Gaussian state-space model (LGSSM) end-to-end with synthetic data, visualizes each stage, and checks that learning works.

## 0) Preflight and Compatibility Check

This cell prints versions and runs a tiny Dynamax sampling smoke test. If it fails with a NumPy 2.x compatibility error (`np.issctype`), use the fallback instructions shown in the output.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, vmap

import dynamax
import 
from dynamax.linear_gaussian_ssm import LinearGaussianSSM

print('numpy:', np.__version__)
print('jax:', jax.__version__)
print('dynamax:', dynamax.__version__)

def smoke_test_lgssm():
    key = random.PRNGKey(0)
    model = LinearGaussianSSM(state_dim=2, emission_dim=1)
    params, _ = model.initialize(key=key)
    states, emissions = model.sample(params, key, num_timesteps=5)
    return states.shape, emissions.shape

try:
    s_shape, y_shape = smoke_test_lgssm()
    print('Smoke test passed. states shape:', s_shape, 'emissions shape:', y_shape)
except Exception as e:
    print('Smoke test failed:', repr(e))
    print('\nFallback (without editing repo files):')
    print('  pip install "numpy<2" "jax[cpu]>=0.4.30" "tensorflow-probability>=0.24" dynamax')
    print('Then restart kernel and rerun.')


numpy: 2.2.6
jax: 0.6.2
dynamax: 1.0.1
Smoke test failed: AttributeError('`np.issctype` was removed in the NumPy 2.0 release. Use `issubclass(rep, np.generic)` instead.')

Fallback (without editing repo files):
  pip install "numpy<2" "jax[cpu]>=0.4.30" "tensorflow-probability>=0.24" dynamax
Then restart kernel and rerun.


## 1) Setup and Plot Helpers

Reusable plotting functions to keep the workflow readable and consistent.

In [6]:
SEED = 7
STATE_DIM = 4
EMISSION_DIM = 2
NUM_SEQS = 64
T = 200
EM_ITERS = 35

np.random.seed(SEED)
key = random.PRNGKey(SEED)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

def to_np(x):
    return np.asarray(x)

def plot_matrix_heatmap(M, title, cmap='coolwarm'):
    M = to_np(M)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(M, cmap=cmap, aspect='auto')
    ax.set_title(title)
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

def plot_timeseries_panel(arr, title, max_dims=4, max_lines=3):
    arr = to_np(arr)  # [N, T, D] or [T, D]
    if arr.ndim == 2:
        arr = arr[None, ...]
    N, TT, D = arr.shape
    d_show = min(D, max_dims)
    n_show = min(N, max_lines)
    fig, axes = plt.subplots(d_show, 1, figsize=(10, 2.2 * d_show), sharex=True)
    if d_show == 1:
        axes = [axes]
    t = np.arange(TT)
    for d in range(d_show):
        for n in range(n_show):
            axes[d].plot(t, arr[n, :, d], lw=1.2, alpha=0.9, label=f'seq {n}' if d == 0 else None)
        axes[d].set_ylabel(f'dim {d}')
    axes[0].set_title(title)
    axes[-1].set_xlabel('time')
    if n_show > 1:
        axes[0].legend(loc='upper right', frameon=True)
    plt.tight_layout()
    plt.show()

def plot_ll_trace(log_probs):
    ll = to_np(log_probs)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(ll, marker='o', lw=1.5)
    axes[0].set_title('EM Log-Likelihood Trace')
    axes[0].set_xlabel('iteration')
    axes[0].set_ylabel('log p(y)')
    delta = np.diff(ll)
    axes[1].plot(delta, marker='o', lw=1.5, color='tab:orange')
    axes[1].axhline(0.0, color='k', ls='--', lw=1)
    axes[1].set_title('Log-Likelihood Increment')
    axes[1].set_xlabel('iteration')
    axes[1].set_ylabel('delta')
    plt.tight_layout()
    plt.show()

def plot_true_vs_learned_params(A_true, A_hat, C_true, C_hat):
    mats = [to_np(A_true), to_np(A_hat), to_np(C_true), to_np(C_hat)]
    titles = ['A true', 'A learned', 'C true', 'C learned']
    fig, axes = plt.subplots(2, 2, figsize=(9, 7))
    for ax, M, title in zip(axes.ravel(), mats, titles):
        im = ax.imshow(M, cmap='coolwarm', aspect='auto')
        ax.set_title(title)
        ax.set_xlabel('Column')
        ax.set_ylabel('Row')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

def plot_reconstruction(y_true, y_pred, title, max_dims=2):
    y_true = to_np(y_true)
    y_pred = to_np(y_pred)
    D = y_true.shape[-1]
    d_show = min(D, max_dims)
    fig, axes = plt.subplots(d_show, 1, figsize=(10, 2.4 * d_show), sharex=True)
    if d_show == 1:
        axes = [axes]
    t = np.arange(y_true.shape[0])
    for d in range(d_show):
        axes[d].plot(t, y_true[:, d], label='observed', lw=1.3)
        axes[d].plot(t, y_pred[:, d], label='predicted', lw=1.3)
        axes[d].set_ylabel(f'dim {d}')
    axes[0].set_title(title)
    axes[-1].set_xlabel('time')
    axes[0].legend()
    plt.tight_layout()
    plt.show()


## 2) Ground-Truth LGSSM and Dynamics Visuals

In [7]:
def make_stable_matrix(key, dim, target_radius=0.92):
    M = random.normal(key, (dim, dim))
    eigvals = jnp.linalg.eigvals(M)
    rho = jnp.max(jnp.abs(eigvals)) + 1e-6
    return (target_radius / rho) * M

def diag_psd(diag_vals):
    return jnp.diag(jnp.array(diag_vals))

key, kA, kC = random.split(key, 3)
A_true = make_stable_matrix(kA, STATE_DIM, target_radius=0.9)
C_true = 0.8 * random.normal(kC, (EMISSION_DIM, STATE_DIM))
Q_true = diag_psd([0.07, 0.05, 0.04, 0.03])
R_true = diag_psd([0.12, 0.08])
m0_true = jnp.zeros((STATE_DIM,))
P0_true = diag_psd([1.0, 1.0, 1.0, 1.0])

true_model = LinearGaussianSSM(STATE_DIM, EMISSION_DIM)
params_true = true_model.initialize(
    key=random.PRNGKey(SEED + 101),
    initial_mean=m0_true,
    initial_covariance=P0_true,
    dynamics_weights=A_true,
    dynamics_covariance=Q_true,
    emission_weights=C_true,
    emission_covariance=R_true,
)[0]

plot_matrix_heatmap(A_true, 'Ground Truth A (state transition)')
plot_matrix_heatmap(C_true, 'Ground Truth C (emission matrix)')
plot_matrix_heatmap(Q_true, 'Ground Truth Q (state noise covariance)', cmap='viridis')
plot_matrix_heatmap(R_true, 'Ground Truth R (emission noise covariance)', cmap='viridis')

eig_A = np.linalg.eigvals(to_np(A_true))
theta = np.linspace(0, 2 * np.pi, 400)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(np.cos(theta), np.sin(theta), 'k--', lw=1, label='unit circle')
ax.scatter(eig_A.real, eig_A.imag, color='tab:red', s=50, label='eig(A)')
ax.set_aspect('equal')
ax.set_title('Eigenvalues of A (stability check)')
ax.set_xlabel('real')
ax.set_ylabel('imag')
ax.legend()
plt.show()


TypeError: LinearGaussianSSM.initialize() got an unexpected keyword argument 'emissions_weights'

In [ ]:
# Impulse-like response: each basis vector as initial condition, noiseless propagation.
steps = 35
basis = np.eye(STATE_DIM)
responses = []
for i in range(STATE_DIM):
    x = basis[i].copy()
    traj = [x.copy()]
    for _ in range(steps - 1):
        x = to_np(A_true) @ x
        traj.append(x.copy())
    responses.append(np.stack(traj, axis=0))
responses = np.stack(responses, axis=0)  # [STATE_DIM, steps, STATE_DIM]

for init_idx in range(min(STATE_DIM, 3)):
    plot_timeseries_panel(
        responses[init_idx:init_idx+1],
        title=f'Noiseless dynamics response from basis init e_{init_idx}',
        max_dims=STATE_DIM,
        max_lines=1,
    )


## 3) Generate Synthetic Data and Visualize It

In [ ]:
keys = random.split(random.PRNGKey(SEED + 202), NUM_SEQS)
states_batch, emissions_batch = vmap(lambda k: true_model.sample(params_true, k, T))(keys)

print('states_batch shape:', states_batch.shape)
print('emissions_batch shape:', emissions_batch.shape)

plot_timeseries_panel(states_batch, 'Sample latent state trajectories', max_dims=STATE_DIM, max_lines=3)
plot_timeseries_panel(emissions_batch, 'Sample emission trajectories', max_dims=EMISSION_DIM, max_lines=3)

y_np = to_np(emissions_batch)
t = np.arange(T)
fig, axes = plt.subplots(EMISSION_DIM, 1, figsize=(10, 2.4 * EMISSION_DIM), sharex=True)
if EMISSION_DIM == 1:
    axes = [axes]
for d in range(EMISSION_DIM):
    mu = y_np[:, :, d].mean(axis=0)
    sd = y_np[:, :, d].std(axis=0)
    axes[d].plot(t, mu, color='tab:blue', lw=1.5, label='mean over sequences')
    axes[d].fill_between(t, mu - sd, mu + sd, color='tab:blue', alpha=0.2, label='±1 std')
    axes[d].set_ylabel(f'dim {d}')
axes[0].set_title('Emission summary across sequences')
axes[-1].set_xlabel('time')
axes[0].legend()
plt.tight_layout()
plt.show()


## 4) Initialize Learnable Model and Show Initial Mismatch

In [ ]:
fit_model = LinearGaussianSSM(STATE_DIM, EMISSION_DIM)
init_params, init_props = fit_model.initialize(key=random.PRNGKey(SEED + 303))

A_init = init_params.dynamics.weights
C_init = init_params.emissions.weights
plot_true_vs_learned_params(A_true, A_init, C_true, C_init)

seq0_y = emissions_batch[0]
post_init = fit_model.smoother(init_params, seq0_y)
xhat_init = post_init.smoothed_means
yhat_init = xhat_init @ init_params.emissions.weights.T
plot_reconstruction(seq0_y, yhat_init, 'Initial model reconstruction on one sequence')


## 5) Train with EM and Visualize Learning Progress

In [ ]:
fitted_params, ll_trace = fit_model.fit_em(init_params, init_props, emissions_batch, num_iters=EM_ITERS, verbose=False)
ll_trace = np.asarray(ll_trace)

print(f'Initial log-likelihood: {ll_trace[0]:.3f}')
print(f'Final log-likelihood:   {ll_trace[-1]:.3f}')
print(f'Improvement:            {ll_trace[-1] - ll_trace[0]:.3f}')

plot_ll_trace(ll_trace)
plot_true_vs_learned_params(
    A_true, fitted_params.dynamics.weights,
    C_true, fitted_params.emissions.weights,
)

A_dist = np.linalg.norm(to_np(A_true) - to_np(fitted_params.dynamics.weights), ord='fro')
C_dist = np.linalg.norm(to_np(C_true) - to_np(fitted_params.emissions.weights), ord='fro')
print(f'||A_true - A_hat||_F = {A_dist:.4f}')
print(f'||C_true - C_hat||_F = {C_dist:.4f}')


## 6) Post-Fit Evaluation: Filtering, Smoothing, Residuals

In [ ]:
held_idx = NUM_SEQS - 1
y_hold = emissions_batch[held_idx]
x_hold_true = states_batch[held_idx]

post_init_hold = fit_model.smoother(init_params, y_hold)
xhat_init_hold = post_init_hold.smoothed_means
yhat_init_hold = xhat_init_hold @ init_params.emissions.weights.T

post_fit_hold = fit_model.smoother(fitted_params, y_hold)
xhat_fit_hold = post_fit_hold.smoothed_means
yhat_fit_hold = xhat_fit_hold @ fitted_params.emissions.weights.T

mse_init = np.mean((to_np(y_hold) - to_np(yhat_init_hold)) ** 2)
mse_fit = np.mean((to_np(y_hold) - to_np(yhat_fit_hold)) ** 2)

print(f'Reconstruction MSE (init): {mse_init:.6f}')
print(f'Reconstruction MSE (fit):  {mse_fit:.6f}')

plot_reconstruction(y_hold, yhat_fit_hold, 'Held-out reconstruction after EM fit')

resid = to_np(y_hold - yhat_fit_hold)
fig, axes = plt.subplots(EMISSION_DIM, 2, figsize=(11, 3 * EMISSION_DIM))
if EMISSION_DIM == 1:
    axes = np.array([axes])
for d in range(EMISSION_DIM):
    axes[d, 0].plot(resid[:, d], lw=1.2)
    axes[d, 0].axhline(0.0, color='k', ls='--', lw=1)
    axes[d, 0].set_title(f'Residual over time (dim {d})')
    axes[d, 0].set_xlabel('time')
    axes[d, 0].set_ylabel('residual')

    axes[d, 1].hist(resid[:, d], bins=25, alpha=0.8, color='tab:green')
    axes[d, 1].set_title(f'Residual histogram (dim {d})')
    axes[d, 1].set_xlabel('residual')
plt.tight_layout()
plt.show()

plot_timeseries_panel(
    np.stack([to_np(x_hold_true), to_np(xhat_fit_hold)], axis=0),
    title='Latent states: true (seq 0 line) vs smoothed estimate (seq 1 line)',
    max_dims=min(STATE_DIM, 4),
    max_lines=2,
)


## 7) Quick Acceptance Checks

These checks are intentionally moderate and practical.

In [ ]:
ll_non_decreasing = np.all(np.diff(ll_trace) >= -1e-6)
print('Log-likelihood non-decreasing (within tolerance):', ll_non_decreasing)
print('EM improved fit:', ll_trace[-1] > ll_trace[0])
print('Reconstruction improved:', mse_fit < mse_init)
